In [3]:
"""
build_natural_resources.py
==========================
Reproducible pipeline to construct natural_resources_final_clean.csv

DATA SOURCES
------------
1. Energy Institute (EI) Statistical Review — Narrow CSV file
   - Oil, Natural Gas, Coal: Production, Consumption, Reserves (where available)
   - Transition minerals: Cobalt, Lithium, Natural Graphite, Rare Earths (Production + Reserves)

2. Energy Institute Excel Workbook (base_dataset.xlsx)
   - Mineral P-R sheets (Production + Reserves by country, annual)
   - Oil crude prices since 1861
   - Coal & Uranium Prices
   - Mineral Commodity Prices (Cobalt, Lithium, Nickel, Natural Graphite)

3. OWID Mineral CSVs (~/Downloads/Minerals/)
   - Individual CSVs named like: copper-production.csv, copper-unit-value.csv
   - Columns: Entity, Code, Year, <metric>|<Resource>|...|<unit>
   - Used to fill gaps not covered by EI (e.g. additional countries, longer time series)

OUTPUT
------
natural_resources_final_clean.csv
  Columns: Country, Year, Value, Resource, Metric
  - Metric values: Production | Reserves | Consumption | Price
  - EI data takes priority over OWID where both exist (deduplicated keep='first')

NOTES
-----
- "Russian Federation" is preserved as-is (matches raw source data)
- EI Excel P-R sheets: production runs 1995-2024, reserves are single-year snapshots
  (at end of 2024) and are broadcast to a single Year=2024 row
- Coal Reserves, Coal Price, Natural Gas Price are NOT in the EI narrow CSV —
  they are extracted from the Excel workbook instead
- OWID files must follow naming convention: <resource-slug>-<metric-slug>.csv
  e.g. copper-production.csv, copper-unit-value.csv
"""

import pandas as pd
import numpy as np
from pathlib import Path
import logging
import re

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# ============================================================
# PATHS — update these to match your local setup
# ============================================================
MINERALS_DIR  = Path.home() / "Downloads" / "Minerals"
EI_CSV_PATH   = Path.home() / "Downloads" / "Statistical Review of World Energy Narrow File-1.csv"
EI_XLSX_PATH  = Path.home() / "Desktop" / "data generic" / "base_dataset.xlsx"
OUTPUT_PATH   = Path.home() / "Downloads" / "natural_resources_final_clean.csv"

# ============================================================
# MAPPINGS
# ============================================================

# EI Narrow CSV: Var code -> (Resource, Metric)
# NOTE: Coal Reserves, Coal Price, Gas Price are absent from the narrow CSV
#       and are sourced from the Excel workbook instead.
EI_CSV_VAR_MAP = {
    "oilprod_kbd":     ("Oil",           "Production"),
    "oilcons_kbd":     ("Oil",           "Consumption"),
    "gasprod_bcm":     ("Natural Gas",   "Production"),
    "gascons_bcm":     ("Natural Gas",   "Consumption"),
    "coalprod_mt":     ("Coal",          "Production"),
    "coalcons_ej":     ("Coal",          "Consumption"),
    "cobalt_kt":       ("Cobalt",        "Production"),
    "cobaltres_kt":    ("Cobalt",        "Reserves"),
    "lithium_kt":      ("Lithium",       "Production"),
    "lithiumres_kt":   ("Lithium",       "Reserves"),
    "graphite_kt":     ("Natural Graphite", "Production"),
    "graphiteres_kt":  ("Natural Graphite", "Reserves"),
    "rareearths_kt":   ("Rare Earth",    "Production"),
    "rareearthsres_kt":("Rare Earth",    "Reserves"),
}

# EI Excel P-R sheet name -> Resource label
EXCEL_PR_SHEETS = {
    "Cobalt P-R":              "Cobalt",
    "Lithium P-R":             "Lithium",
    "Natural Graphite P-R":    "Natural Graphite",
    "Rare Earth metals P-R":   "Rare Earth",
    "Copper P-R":              "Copper",
    "Manganese P-R":           "Manganese",
    "Nickel P-R":              "Nickel",
    "Zinc P-R":                "Zinc",
    "Platinum Group Metals P-R": "Platinum Group",
    "Bauxite P-R":             "Bauxite",
    "Aluminium P-R":           "Aluminium",
    "Tin P-R":                 "Tin",
    "Vanadium P-R":            "Vanadium",
}

# OWID folder name -> (Resource, Metric)
# Each folder contains a <folder-name>.csv inside it.
# Files with dots in the name (e.g. zinc-unit-value.filtered) are handled by inference.
OWID_FILE_MAP = {
    "aluminum-production":                        ("Aluminium",           "Production"),
    "aluminum-unit-value":                        ("Aluminium",           "Price"),
    "bauxite-production":                         ("Bauxite",             "Production"),
    "bauxite-unit-value":                         ("Bauxite",             "Price"),
    "coal-production":                            ("Coal",                "Production"),
    "cobalt-production":                          ("Cobalt",              "Production"),
    "cobalt-reserves":                            ("Cobalt",              "Reserves"),
    "cobalt-unit-value":                          ("Cobalt",              "Price"),
    "copper-production":                          ("Copper",              "Production"),
    "copper-reserves":                            ("Copper",              "Reserves"),
    "copper-unit-value":                          ("Copper",              "Price"),
    "gold-production":                            ("Gold",                "Production"),
    "gold-reserves":                              ("Gold",                "Reserves"),
    "gold-reserves.filtered":                     ("Gold",                "Reserves"),
    "gold-unit-value":                            ("Gold",                "Price"),
    "graphite-production":                        ("Natural Graphite",    "Production"),
    "graphite-reserves":                          ("Natural Graphite",    "Reserves"),
    "graphite-unit-value":                        ("Natural Graphite",    "Price"),
    "iron-ore-crude-ore-production":              ("Iron ore",            "Production"),
    "iron-ore-crude-ore-unit-value":              ("Iron ore",            "Price"),
    "lead-production":                            ("Lead",                "Production"),
    "lead-reserves":                              ("Lead",                "Reserves"),
    "lead-unit-value":                            ("Lead",                "Price"),
    "lithium-production":                         ("Lithium",             "Production"),
    "lithium-reserves":                           ("Lithium",             "Reserves"),
    "lithium-unit-value":                         ("Lithium",             "Price"),
    "magnesium-compounds-production":             ("Magnesium compounds", "Production"),
    "magnesium-compounds-unit-value":             ("Magnesium compounds", "Price"),
    "manganese-production":                       ("Manganese",           "Production"),
    "manganese-reserves":                         ("Manganese",           "Reserves"),
    "manganese-unit-value":                       ("Manganese",           "Price"),
    "nickel-production":                          ("Nickel",              "Production"),
    "nickel-reserves":                            ("Nickel",              "Reserves"),
    "nickel-unit-value":                          ("Nickel",              "Price"),
    "petroleum-production":                       ("Oil",                 "Production"),
    "platinum-group-metals-palladium-production": ("Platinum Group",      "Production"),
    "platinum-group-metals-palladium-reserves":   ("Platinum Group",      "Reserves"),
    "rare-earths-production":                     ("Rare Earth",          "Production"),
    "rare-earths-reserves":                       ("Rare Earth",          "Reserves"),
    "rare-earths-unit-value":                     ("Rare Earth",          "Price"),
    "refined-cadmium-production":                 ("Cadmium",             "Production"),
    "refined-cadmium-production-1":               ("Cadmium",             "Production"),
    "refined-cadmium-unit-value":                 ("Cadmium",             "Price"),
    "silver-production":                          ("Silver",              "Production"),
    "silver-reserves":                            ("Silver",              "Reserves"),
    "silver-reserves-1":                          ("Silver",              "Reserves"),
    "silver-unit-value":                          ("Silver",              "Price"),
    "tin-production":                             ("Tin",                 "Production"),
    "tin-reserves":                               ("Tin",                 "Reserves"),
    "tin-unit-value":                             ("Tin",                 "Price"),
    "vanadium-production":                        ("Vanadium",            "Production"),
    "vanadium-reserves":                          ("Vanadium",            "Reserves"),
    "vanadium-unit-value":                        ("Vanadium",            "Price"),
    "zinc-production":                            ("Zinc",                "Production"),
    "zinc-reserves":                              ("Zinc",                "Reserves"),
    "zinc-unit-value.filtered":                   ("Zinc",                "Price"),
}

# OWID column name patterns -> Metric override (if slug map isn't enough)
OWID_METRIC_PATTERNS = {
    "production":  "Production",
    "unit value":  "Price",
    "reserves":    "Reserves",
    "consumption": "Consumption",
}

FINAL_COLS = ["Country", "Year", "Value", "Resource", "Metric"]

# ============================================================
# SOURCE 1: EI Narrow CSV
# ============================================================

def load_ei_csv(path: Path) -> pd.DataFrame:
    """Extract production/consumption/reserves from the EI narrow CSV."""
    logger.info("Loading EI Narrow CSV...")
    if not path.exists():
        logger.warning(f"EI CSV not found: {path}")
        return pd.DataFrame(columns=FINAL_COLS)

    df = pd.read_csv(path)
    frames = []
    for var_code, (resource, metric) in EI_CSV_VAR_MAP.items():
        subset = df[df["Var"] == var_code][["Country", "Year", "Value"]].copy()
        if subset.empty:
            logger.warning(f"  Var code not found in EI CSV: {var_code}")
            continue
        subset["Resource"] = resource
        subset["Metric"]   = metric
        frames.append(subset)

    if not frames:
        return pd.DataFrame(columns=FINAL_COLS)

    result = pd.concat(frames, ignore_index=True)
    logger.info(f"  EI CSV: {len(result):,} rows across {len(frames)} variables")
    return result[FINAL_COLS]


# ============================================================
# SOURCE 2a: EI Excel — Mineral P-R sheets
# ============================================================

def _parse_pr_sheet(xl: pd.ExcelFile, sheet: str, resource: str) -> pd.DataFrame:
    """
    Parse a mineral Production-and-Reserves sheet from the EI Excel workbook.

    Sheet layout (0-indexed rows):
      Row 0: Title
      Row 1: Section labels ("Mine production" / "Reserves")
      Row 2: Units row + year headers (columns 1..N are years, last few are stats)
      Row 3: blank
      Rows 4..N-2: Country data
      Last rows: footnotes / totals (excluded)

    Reserves are a single snapshot column labelled "At end of <year>" in row 2.
    """
    df = pd.read_excel(xl, sheet_name=sheet, header=None)

    # --- Find year columns (numeric values in row 2) ---
    header_row = df.iloc[2]
    year_cols = {}
    for col_idx, val in enumerate(header_row):
        try:
            yr = int(float(val))
            if 1900 < yr < 2100:
                year_cols[col_idx] = yr
        except (ValueError, TypeError):
            pass

    # --- Find reserves column (header row 1 contains "Reserves") ---
    reserves_col = None
    reserves_year = None
    for col_idx, val in enumerate(df.iloc[1]):
        if isinstance(val, str) and "reserves" in val.lower():
            reserves_col = col_idx
            # Year is in row 2 for that column, or parse from label in row 2
            break
    if reserves_col is not None:
        # The year label for reserves is usually in row 2 of that col ("At end of 2024")
        res_label = str(df.iloc[2, reserves_col])
        match = re.search(r"(\d{4})", res_label)
        reserves_year = int(match.group(1)) if match else None

    # --- Identify data rows: col 0 is a country name, not NaN, not Total/Source ---
    skip_patterns = re.compile(
        r"(total|source|rest of|^\♦|growth|share|r/p|billion|million|thousand|"
        r"^nan$|^\s*$|production|reserves|mine|content)",
        re.IGNORECASE,
    )
    data_rows = []
    for i, row in df.iterrows():
        val = str(row.iloc[0]).strip()
        if i <= 3:
            continue
        if skip_patterns.search(val):
            continue
        data_rows.append(i)

    frames = []

    # --- Production: melt year columns ---
    if year_cols and data_rows:
        prod_df = df.iloc[data_rows, [0] + list(year_cols.keys())].copy()
        prod_df.columns = ["Country"] + [year_cols[c] for c in year_cols.keys()]
        prod_df = prod_df.melt(id_vars="Country", var_name="Year", value_name="Value")
        prod_df["Resource"] = resource
        prod_df["Metric"]   = "Production"
        frames.append(prod_df)

    # --- Reserves: single snapshot column ---
    if reserves_col is not None and reserves_year is not None and data_rows:
        res_df = df.iloc[data_rows, [0, reserves_col]].copy()
        res_df.columns = ["Country", "Value"]
        res_df["Year"]     = reserves_year
        res_df["Resource"] = resource
        res_df["Metric"]   = "Reserves"
        frames.append(res_df)

    if not frames:
        logger.warning(f"  No data parsed from sheet: {sheet}")
        return pd.DataFrame(columns=FINAL_COLS)

    out = pd.concat(frames, ignore_index=True)
    return out[FINAL_COLS]


def load_ei_excel_minerals(path: Path) -> pd.DataFrame:
    """Parse all mineral P-R sheets from the EI Excel workbook."""
    logger.info("Loading EI Excel mineral P-R sheets...")
    if not path.exists():
        logger.warning(f"EI Excel not found: {path}")
        return pd.DataFrame(columns=FINAL_COLS)

    xl = pd.ExcelFile(path)
    frames = []
    for sheet, resource in EXCEL_PR_SHEETS.items():
        if sheet not in xl.sheet_names:
            logger.warning(f"  Sheet not found: {sheet}")
            continue
        try:
            parsed = _parse_pr_sheet(xl, sheet, resource)
            frames.append(parsed)
            logger.info(f"  {sheet}: {len(parsed):,} rows")
        except Exception as e:
            logger.error(f"  Failed to parse {sheet}: {e}")

    if not frames:
        return pd.DataFrame(columns=FINAL_COLS)
    return pd.concat(frames, ignore_index=True)


# ============================================================
# SOURCE 2b: EI Excel — Price sheets
# ============================================================

def load_ei_excel_prices(path: Path) -> pd.DataFrame:
    """
    Extract price time series from EI Excel workbook.
    Covers: Oil (crude, $/bbl), Coal (NW Europe benchmark, $/tonne),
            Mineral Commodity Prices (Cobalt, Lithium, Nickel, Natural Graphite).
    """
    logger.info("Loading EI Excel price sheets...")
    if not path.exists():
        return pd.DataFrame(columns=FINAL_COLS)

    xl = pd.ExcelFile(path)
    frames = []

    # --- Oil crude prices (world price, not country-level) ---
    if "Oil crude prices since 1861" in xl.sheet_names:
        df = pd.read_excel(xl, sheet_name="Oil crude prices since 1861", header=None)
        # Row 3: Year | $ money of the day | $ 2024
        # Data starts row 4
        price_df = df.iloc[4:, [0, 2]].copy()
        price_df.columns = ["Year", "Value"]
        price_df = price_df.dropna()
        price_df["Country"]  = "World"
        price_df["Resource"] = "Oil"
        price_df["Metric"]   = "Price"
        frames.append(price_df[FINAL_COLS])
        logger.info(f"  Oil prices: {len(price_df):,} rows")

    # --- Coal prices: NW Europe benchmark ---
    if "Coal & Uranium - Prices" in xl.sheet_names:
        df = pd.read_excel(xl, sheet_name="Coal & Uranium - Prices", header=None)
        # Row 3: col 0 = Year, col 3 = "Northwest Europe3"
        # Data from row 4 onward
        coal_df = df.iloc[4:, [0, 3]].copy()
        coal_df.columns = ["Year", "Value"]
        coal_df = coal_df[pd.to_numeric(coal_df["Year"], errors="coerce").notna()]
        coal_df["Value"] = pd.to_numeric(coal_df["Value"], errors="coerce")
        coal_df = coal_df.dropna(subset=["Value"])
        coal_df["Country"]  = "World"
        coal_df["Resource"] = "Coal"
        coal_df["Metric"]   = "Price"
        frames.append(coal_df[FINAL_COLS])
        logger.info(f"  Coal prices: {len(coal_df):,} rows")

    # --- Mineral Commodity Prices (Cobalt, Lithium carbonate, Nickel, Natural Graphite) ---
    if "Mineral Commodity Prices" in xl.sheet_names:
        df = pd.read_excel(xl, sheet_name="Mineral Commodity Prices", header=None)
        # Row 3: Year | Cobalt | Lithium carbonate | Nickel Sulphate | ... | Natural Graphite
        # Data from row 4
        headers = df.iloc[3].tolist()
        resource_col_map = {}
        for col_idx, h in enumerate(headers):
            h_str = str(h).lower()
            if "cobalt" in h_str:
                resource_col_map[col_idx] = "Cobalt"
            elif "lithium" in h_str:
                resource_col_map[col_idx] = "Lithium"
            elif "nickel" in h_str:
                resource_col_map[col_idx] = "Nickel"
            elif "graphite" in h_str:
                resource_col_map[col_idx] = "Natural Graphite"

        data = df.iloc[4:].copy()
        year_col = data.iloc[:, 0]
        for col_idx, resource in resource_col_map.items():
            sub = pd.DataFrame({
                "Year":  year_col,
                "Value": data.iloc[:, col_idx],
            })
            sub["Year"]  = pd.to_numeric(sub["Year"], errors="coerce")
            sub["Value"] = pd.to_numeric(sub["Value"], errors="coerce")
            sub = sub.dropna()
            sub["Country"]  = "World"
            sub["Resource"] = resource
            sub["Metric"]   = "Price"
            frames.append(sub[FINAL_COLS])
        logger.info(f"  Mineral commodity prices: {sum(len(f) for f in frames[-len(resource_col_map):]):,} rows")

    if not frames:
        return pd.DataFrame(columns=FINAL_COLS)
    return pd.concat(frames, ignore_index=True)


# ============================================================
# SOURCE 3: OWID Mineral CSVs
# ============================================================

def _infer_metric_from_column(col_name: str) -> str:
    """Infer Metric label from OWID column header string."""
    col_lower = col_name.lower()
    for pattern, metric in OWID_METRIC_PATTERNS.items():
        if pattern in col_lower:
            return metric
    return "Production"  # safe default


def _infer_resource_from_column(col_name: str) -> str:
    """Extract resource name from OWID pipe-delimited column header."""
    # Format: "production|Copper|Mine|tonnes"
    parts = col_name.split("|")
    if len(parts) >= 2:
        return parts[1].strip().title()
    return col_name


def load_owid_csv(filepath: Path, resource: str, metric: str) -> pd.DataFrame:
    """Load a single OWID CSV and return long-format DataFrame."""
    try:
        df = pd.read_csv(filepath)
    except Exception as e:
        logger.warning(f"  Could not read {filepath.name}: {e}")
        return pd.DataFrame(columns=FINAL_COLS)

    if df.empty:
        return pd.DataFrame(columns=FINAL_COLS)

    # Identify value column (everything that isn't Entity/Code/Year)
    value_cols = [c for c in df.columns if c not in ("Entity", "Code", "Year")]
    if not value_cols:
        logger.warning(f"  No value column found in {filepath.name}")
        return pd.DataFrame(columns=FINAL_COLS)

    # Use first value column
    value_col = value_cols[0]

    out = df[["Entity", "Year", value_col]].copy()
    out.columns = ["Country", "Year", "Value"]
    out["Resource"] = resource
    out["Metric"]   = metric
    return out[FINAL_COLS]


def load_all_owid(minerals_dir: Path) -> pd.DataFrame:
    """
    Load all OWID mineral files from the Minerals directory.

    Structure: each resource is a *subfolder* (e.g. copper-production/) containing:
      - copper-production.csv       <- the data file we want
      - copper-production.metadata.json
      - README.md

    File matching strategy:
    1. Folder name matches OWID_FILE_MAP -> use mapped (Resource, Metric) labels
    2. Otherwise, infer from the value column header of the CSV inside
    """
    logger.info("Loading OWID mineral CSVs...")
    if not minerals_dir.exists():
        logger.warning(f"Minerals directory not found: {minerals_dir}")
        return pd.DataFrame(columns=FINAL_COLS)

    # Each entry in Minerals/ is a subfolder; find the CSV inside each one
    subfolders = [p for p in sorted(minerals_dir.iterdir()) if p.is_dir()]
    if not subfolders:
        logger.warning(f"No subfolders found in {minerals_dir}")
        return pd.DataFrame(columns=FINAL_COLS)

    frames = []
    skipped = []
    for folder in subfolders:
        folder_name = folder.name  # e.g. "copper-production"

        # Find the data CSV inside the folder (same name as folder, .csv extension)
        csv_path = folder / f"{folder_name}.csv"
        if not csv_path.exists():
            # Fallback: any CSV in the folder that is not metadata
            candidates = [f for f in folder.glob("*.csv") if "metadata" not in f.name.lower()]
            if not candidates:
                skipped.append(folder_name)
                continue
            csv_path = candidates[0]

        # Resolve (Resource, Metric) from folder name
        if folder_name in OWID_FILE_MAP:
            resource, metric = OWID_FILE_MAP[folder_name]
        else:
            try:
                df = pd.read_csv(csv_path, nrows=1)
                value_cols = [c for c in df.columns if c not in ("Entity", "Code", "Year")]
                if not value_cols:
                    skipped.append(folder_name)
                    continue
                resource = _infer_resource_from_column(value_cols[0])
                metric   = _infer_metric_from_column(value_cols[0])
            except Exception:
                skipped.append(folder_name)
                continue

        parsed = load_owid_csv(csv_path, resource, metric)
        if not parsed.empty:
            frames.append(parsed)
            logger.info(f"  {folder_name}: {len(parsed):,} rows ({resource} | {metric})")
        else:
            skipped.append(folder_name)

    if skipped:
        logger.info(f"  Skipped (empty/unreadable): {skipped}")

    if not frames:
        return pd.DataFrame(columns=FINAL_COLS)
    return pd.concat(frames, ignore_index=True)


# ============================================================
# COMBINE & CLEAN
# ============================================================

def combine_and_clean(*sources: pd.DataFrame) -> pd.DataFrame:
    """
    Merge all sources, clean types, and deduplicate.
    Priority order: EI CSV > EI Excel > OWID (first source wins on duplicates).
    """
    logger.info("Combining and cleaning all sources...")
    combined = pd.concat(list(sources), ignore_index=True)

    # Type coercion
    combined["Value"] = pd.to_numeric(combined["Value"], errors="coerce")
    combined["Year"]  = pd.to_numeric(combined["Year"],  errors="coerce")
    combined = combined.dropna(subset=["Value", "Year", "Country", "Resource", "Metric"])
    combined["Year"] = combined["Year"].astype(int)

    # Strip whitespace
    for col in ["Country", "Resource", "Metric"]:
        combined[col] = combined[col].str.strip()

    # Remove aggregates that pollute country-level analysis
    # Exception: keep "World" rows for Price metric (prices are global benchmarks, not country-level)
    agg_patterns = re.compile(
        r"(^total|^rest of|opec|oecd|european union|^eu$)", re.IGNORECASE
    )
    is_agg = combined["Country"].str.match(agg_patterns, na=False)
    is_world_price = (combined["Country"] == "World") & (combined["Metric"] == "Price")
    combined = combined[~is_agg | is_world_price]

    # Deduplicate: keep first (priority order preserved by concat order)
    before = len(combined)
    combined = combined.drop_duplicates(
        subset=["Country", "Year", "Resource", "Metric"], keep="first"
    )
    logger.info(f"  Deduplication removed {before - len(combined):,} rows")

    return combined.sort_values(["Resource", "Country", "Year"]).reset_index(drop=True)


# ============================================================
# VALIDATION
# ============================================================

def validate(df: pd.DataFrame) -> None:
    logger.info("\n--- Validation Report ---")
    logger.info(f"Total rows:    {len(df):,}")
    logger.info(f"Countries:     {df['Country'].nunique()}")
    logger.info(f"Resources:     {sorted(df['Resource'].unique())}")
    logger.info(f"Metrics:       {sorted(df['Metric'].unique())}")
    logger.info(f"Year range:    {df['Year'].min()} – {df['Year'].max()}")

    russia_check = df[df["Country"].isin(["Russia", "Russian Federation"])]
    logger.info(f"Russia rows:   {len(russia_check):,} (as 'Russian Federation': "
                f"{len(russia_check[russia_check['Country']=='Russian Federation']):,})")

    # Check which EI CSV vars are represented
    expected = {(r, m) for r, m in EI_CSV_VAR_MAP.values()}
    found     = set(zip(df["Resource"], df["Metric"]))
    missing   = expected - found
    if missing:
        logger.warning(f"Expected resource-metric pairs missing from output: {missing}")
    logger.info("-------------------------\n")


# ============================================================
# MAIN
# ============================================================

def main():
    logger.info("=" * 55)
    logger.info("BUILD: natural_resources_final_clean.csv")
    logger.info("=" * 55)

    # Load all sources (priority: EI CSV > EI Excel minerals > EI Excel prices > OWID)
    ei_csv      = load_ei_csv(EI_CSV_PATH)
    ei_minerals = load_ei_excel_minerals(EI_XLSX_PATH)
    ei_prices   = load_ei_excel_prices(EI_XLSX_PATH)
    owid        = load_all_owid(MINERALS_DIR)

    # Combine (order = priority)
    final = combine_and_clean(ei_csv, ei_minerals, ei_prices, owid)

    # Validate
    validate(final)

    # Save
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    final.to_csv(OUTPUT_PATH, index=False)
    logger.info(f"Saved {len(final):,} rows to:\n  {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

INFO: =======================================================
INFO: BUILD: natural_resources_final_clean.csv
INFO: =======================================================
INFO: Loading EI Narrow CSV...
INFO:   EI CSV: 28,473 rows across 14 variables
INFO: Loading EI Excel mineral P-R sheets...
INFO:   Cobalt P-R: 429 rows
INFO:   Lithium P-R: 297 rows
INFO:   Natural Graphite P-R: 429 rows
INFO:   Rare Earth metals P-R: 297 rows
INFO:   Copper P-R: 112 rows
INFO:   Manganese P-R: 98 rows
INFO:   Nickel P-R: 120 rows
INFO:   Zinc P-R: 105 rows
INFO:   Platinum Group Metals P-R: 98 rows
INFO:   Bauxite P-R: 126 rows
INFO:   Aluminium P-R: 91 rows
INFO:   Tin P-R: 126 rows
INFO:   Vanadium P-R: 98 rows
INFO: Loading EI Excel price sheets...
INFO:   Oil prices: 164 rows
INFO:   Coal prices: 38 rows
INFO:   Mineral commodity prices: 80 rows
INFO: Loading OWID mineral CSVs...
INFO:   aluminum-production: 2,420 rows (Aluminium | Production)
INFO:   aluminum-unit-value: 122 rows (Aluminium | P